# 4.2 Creating a RunPod Container

In [1]:
!pip install requests aiohttp pandas tqdm lm-eval -q
# plus whatever benchmark libs you use (e.g. lm-eval==0.4.2)


  Using cached jsonlines-4.0.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached rouge_score-0.1.2-py3-none-any.whl
  Using cached sqlitedict-2.1.0-py3-none-any.whl
  Using cached tqdm_multiprocess-0.0.11-py3-none-any.whl.metadata (5.7 kB)
  Using cached word2number-1.1-py3-none-any.whl
  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached chardet-5.2.0-py3-none-any.whl.metadata (3.4 kB)
   ---------------------------------------- 0.0/7.5 MB ? eta -:--:--
   ------------------------- -------------- 4.7/7.5 MB 23.8 MB/s eta 0:00:01
   ---------------------------------------- 7.5/7.5 MB 23.3 MB/s  0:00:00
   ---------------------------------------- 0.0/8.7 MB ? eta -:--:--
   ------------------------- -------------- 5.5/8.7 MB 27.9 MB/s eta 0:00:01
   -----------------------

## Setup Check

In [8]:
import os
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")

# attempt 2

Install WSL and WSL extension in VScode.

Open terminal for WSL in vscode

Sign in to WSL.

If you forgot your password, this is the process to reset it:

You cannot recover your existing WSL password - passwords are encrypted and not retrievable. However, you have a few options:
Option 1: Reset Your WSL Password (Easiest)
From Windows Command Prompt (as Administrator):
cmd
```# Replace "username" with your actual WSL username
wsl -u root passwd username```
This will prompt you to set a new password.

Then run :

wget -qO- cli.runpod.net | sudo bash

In [1]:
import requests

headers = {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer YOUR_API_KEY'
}

data = {
    'input': {"prompt":"Your prompt"}
}

response = requests.post('https://api.runpod.ai/v2/s3jhcgppyl66io/run', headers=headers, json=data)

In [2]:
response

<Response [401]>

In [6]:
import os
import requests
import json

# Read API key from environment
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
if not RUNPOD_API_KEY:
    raise RuntimeError("RUNPOD_API_KEY is not set in your environment")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {RUNPOD_API_KEY}"   # <— use the env var here
}

data = {
    "input": {"prompt": "Your prompt"}
}

resp = requests.post(
    "https://api.runpod.ai/v2/s3jhcgppyl66io/runsync",
    headers=headers,
    json=data
)

# Print response cleanly
print("HTTP status:", resp.status_code)
print("Raw text:\n", resp.text)

# If response is JSON, pretty-print it
try:
    print("Parsed JSON:\n", json.dumps(resp.json(), indent=2))
except ValueError:
    print("Response is not valid JSON.")


HTTP status: 200
Raw text:
 {"delayTime":26197,"executionTime":2950,"id":"sync-34c820bc-6080-4634-a9d2-e38ff01a7177-u2","output":[{"choices":[{"finish_reason":"stop","index":0,"text":"As the instruction lacks specific content, I'd create a general completion for an open-ended task. Here is one possible scenario: \"I require assistance in forming responses to various customer service situations using empathetic and effective communication techniques.\" To address this need, you can start by learning about key emotional intelligence skills that are essential when dealing with customers such as active listening, showing understanding, maintaining a calm demeanor, asking clarifying questions without placing blame where it's not needed, providing solutions or alternatives to problems raised, and ending on positive or constructive notes. Role-play exercises can also be beneficial in practicing these skills; perhaps you could find scripted dialogues that mirror common customer interactions fo

## trying it with lmeval and serveless endpoints

In [7]:
### updated version to match the MCQ setup
YAML_syseng_string = '''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Question: {{question}}
A. {{choiceA}}
B. {{choiceB}}
C. {{choiceC}}
D. {{choiceD}}
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [9]:
import os
import requests
import json

# 1️⃣ Read API key from environment
RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
if not RUNPOD_API_KEY:
    raise RuntimeError("RUNPOD_API_KEY is not set in your environment")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {RUNPOD_API_KEY}"
}

# 2️⃣ Compose the lm_eval command you want Runpod to execute
#    Adjust model_name, tasks, and other flags as needed.
# model_name = "hf.co/bartowski/Meta-Llama-3.1-70B-Instruct-GGUF:Q4_K_L"
model_name = "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q3_K_L"
lm_eval_command = (
    "lm_eval "
    "--model hf "
    f"--model_args pretrained={model_name} "
    "--include_path ./ "
    "--tasks sysengbench "
    "--device cuda "
    "--output output/sysengbench/ "
    "--log_samples "
    "--num_fewshot 0 "
    "--batch_size auto "
    "--gen_kwargs temperature=0.0"
)

# 3️⃣ JSON payload sent to Runpod
data = {
    "input": {
        "command": lm_eval_command
    }
}

# 4️⃣ Call the serverless endpoint
resp = requests.post(
    "https://api.runpod.ai/v2/s3jhcgppyl66io/runsync",  # replace with your endpoint id
    headers=headers,
    json=data,
    timeout=3600  # increase if your benchmark takes a long time
)

# 5️⃣ Inspect the response
print("HTTP status:", resp.status_code)
print("Raw text:\n", resp.text)

# If response is JSON, pretty-print and save
try:
    resp_json = resp.json()
    print("Parsed JSON:\n", json.dumps(resp_json, indent=2))

    # Optionally save stdout to a file
    if "output" in resp_json and "stdout" in resp_json["output"]:
        with open("lm_eval_stdout.txt", "w", encoding="utf-8") as f:
            f.write(resp_json["output"]["stdout"])
    if "output" in resp_json and "stderr" in resp_json["output"]:
        with open("lm_eval_stderr.txt", "w", encoding="utf-8") as f:
            f.write(resp_json["output"]["stderr"])
except ValueError:
    print("Response is not valid JSON.")


HTTP status: 200
Raw text:
 {"delayTime":639,"error":"Error code: 400 - {'error': {'message': \"[] is too short - 'messages'\", 'type': 'invalid_request_error', 'param': None, 'code': None}}","executionTime":343,"id":"sync-1d4d094e-e06c-4d9b-be6b-5c2d626f0498-u2","status":"FAILED","workerId":"nwwj18htt54z5a"}

Parsed JSON:
 {
  "delayTime": 639,
  "error": "Error code: 400 - {'error': {'message': \"[] is too short - 'messages'\", 'type': 'invalid_request_error', 'param': None, 'code': None}}",
  "executionTime": 343,
  "id": "sync-1d4d094e-e06c-4d9b-be6b-5c2d626f0498-u2",
  "status": "FAILED",
  "workerId": "nwwj18htt54z5a"
}


In [ ]:
!lm_eval \
    --model local-chat-completions \
    --model_args model='hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q3_K_L',base_url='https://api.runpod.ai/v2/s3jhcgppyl66io/runsync' \
    --include_path ./ \
    --tasks sysengbench \
    --device cpu \
    --output output/sysengbench/ \
    --log_samples \
    --limit 10 \
    --batch_size 1 \
    --gen_kwargs temperature=0.0 \
    --apply_chat_template


In [2]:
!pip install sseclient-py requests -q

In [4]:
import os
import requests
from sseclient import SSEClient   # SSE (server-sent events) client

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
if not RUNPOD_API_KEY:
    raise RuntimeError("RUNPOD_API_KEY is not set in your environment")

ENDPOINT_ID = "s3jhcgppyl66io"  # replace with your Runpod serverless endpoint id

url = f"https://api.runpod.ai/v2/{ENDPOINT_ID}/stream"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {RUNPOD_API_KEY}",
}

# Example OpenAI-style request body
payload = {
    "model": "your-model-name",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a limerick about GPUs."}
    ],
    "stream": True
}

# Create a streaming POST request
response = requests.post(url, headers=headers, json=payload, stream=True)

# Use SSEClient to iterate over events as they arrive
client = SSEClient(response)
for event in client.events():
    # Each event.data is a JSON string with a 'choices' list if following OpenAI format
    print(event.data)


In [10]:
import os, asyncio, aiohttp, json

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"
IMAGE_NAME = "runpod/ollama:latest"
GPU_TYPE = "A40"  # ✅ Use correct Runpod GPU type string

async def create_and_check_pod():
    async with aiohttp.ClientSession() as session:
        query = f"""
        mutation {{
          podCreate(input: {{
            name: "hello-ollama",
            imageName: "{IMAGE_NAME}",
            gpuTypeId: "{GPU_TYPE}",
            containerDiskInGb: 20,
            ports: [{{containerPort:11434, hostPort:11434, protocol:TCP}}],
            env: [{{key:"OLLAMA_HOST", value:"0.0.0.0"}}]
          }}) {{
            id
            status
          }}
        }}
        """
        async with session.post(
            RUNPOD_ENDPOINT,
            headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            json={"query": query}
        ) as r:
            resp = await r.json()
            print("Raw response:", json.dumps(resp, indent=2))   # 🟢 Debug print
            if "errors" in resp:
                raise RuntimeError(f"GraphQL error: {resp['errors']}")
            if "data" not in resp or "podCreate" not in resp["data"]:
                raise RuntimeError(f"Unexpected response: {resp}")
            pod_id = resp["data"]["podCreate"]["id"]
            print(f"Pod created: {pod_id}")
            return pod_id

await create_and_check_pod()


Raw response: {
  "errors": [
    {
      "message": "Cannot query field \"podCreate\" on type \"Mutation\". Did you mean \"podReset\", \"podResume\", or \"teamCreate\"?",
      "locations": [
        {
          "line": 3,
          "column": 11
        }
      ],
      "extensions": {
        "code": "GRAPHQL_VALIDATION_FAILED"
      }
    }
  ]
}


RuntimeError: GraphQL error: [{'message': 'Cannot query field "podCreate" on type "Mutation". Did you mean "podReset", "podResume", or "teamCreate"?', 'locations': [{'line': 3, 'column': 11}], 'extensions': {'code': 'GRAPHQL_VALIDATION_FAILED'}}]

In [11]:
import os, asyncio, aiohttp, json

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"
IMAGE_NAME = "runpod/ollama:latest"
GPU_TYPE = "NVIDIA A40"  # or whatever valid GPU type ID according to your account

async def create_and_check_pod():
    async with aiohttp.ClientSession() as session:
        query = f"""
        mutation {{
          podFindAndDeployOnDemand(input: {{
            name: "hello-ollama",
            imageName: "{IMAGE_NAME}",
            gpuTypeId: "{GPU_TYPE}",
            gpuCount: 1,
            containerDiskInGb: 20,
            volumeInGb: 0,      # or set to nonzero if you want persistent volume
            ports: "11434/tcp",
            env: [{{ key: "OLLAMA_HOST", value: "0.0.0.0" }}]
          }}) {{
            id
            status
          }}
        }}
        """
        async with session.post(
            RUNPOD_ENDPOINT,
            headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            json={"query": query}
        ) as resp:
            data = await resp.json()
            print("Raw response:", json.dumps(data, indent=2))
            if "errors" in data:
                raise RuntimeError(f"GraphQL error: {data['errors']}")
            pod_info = data.get("data", {}).get("podFindAndDeployOnDemand")
            if not pod_info:
                raise RuntimeError(f"No pod info in response: {data}")
            pod_id = pod_info["id"]
            print(f"Pod created: {pod_id}")

        # Poll for status
        check_query = f"""query {{
          pod(id: "{pod_id}") {{
            status
          }}
        }}"""
        while True:
            async with session.post(
                RUNPOD_ENDPOINT,
                headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
                json={"query": check_query}
            ) as resp:
                sta = await resp.json()
                print("Status check:", json.dumps(sta, indent=2))
                status = sta.get("data", {}).get("pod", {}).get("status")
                print("Current status:", status)
                if status == "RUNNING":
                    break
            await asyncio.sleep(5)
        print(f"✅ Pod {pod_id} is RUNNING!")
        return pod_id

# In Jupyter, top level:
# either use nest_asyncio or use await
# with nest_asyncio:
# import nest_asyncio; nest_asyncio.apply()

pod_id = await create_and_check_pod()


Raw response: {
  "errors": [
    {
      "message": "Cannot query field \"status\" on type \"Pod\".",
      "locations": [
        {
          "line": 14,
          "column": 13
        }
      ],
      "extensions": {
        "code": "GRAPHQL_VALIDATION_FAILED"
      }
    }
  ]
}


RuntimeError: GraphQL error: [{'message': 'Cannot query field "status" on type "Pod".', 'locations': [{'line': 14, 'column': 13}], 'extensions': {'code': 'GRAPHQL_VALIDATION_FAILED'}}]

In [13]:
import os, asyncio, aiohttp, json

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"
IMAGE_NAME = "runpod/ollama:latest"
GPU_TYPE = "A40"  # check via gpuTypes query

async def create_and_check_pod():
    async with aiohttp.ClientSession() as session:
        # Mutation to create pod
        query = f"""
        mutation {{
          podFindAndDeployOnDemand(input: {{
            name: "hello-ollama",
            imageName: "{IMAGE_NAME}",
            gpuTypeId: "{GPU_TYPE}",
            gpuCount: 1,
            containerDiskInGb: 20,
            volumeInGb: 0,
            ports: ["11434/tcp"],
            env: [{{ key: "OLLAMA_HOST", value: "0.0.0.0" }}]
          }}) {{
            id
            # maybe other immediate info
          }}
        }}
        """
        async with session.post(
            RUNPOD_ENDPOINT,
            headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            json={"query": query}
        ) as resp:
            data = await resp.json()
            print("Create raw response:", json.dumps(data, indent=2))
            if "errors" in data:
                raise RuntimeError(f"Create error: {data['errors']}")
            pod_info = data.get("data", {}).get("podFindAndDeployOnDemand")
            if not pod_info:
                raise RuntimeError(f"No pod info: {data}")
            pod_id = pod_info["id"]
            print(f"Pod created: {pod_id}")

        # Poll the pod
        check_query = f"""
        query {{
          pod(input: {{ podId: "{pod_id}" }}) {{
            id
            runtime {{
              uptimeInSeconds
              ports {{
                publicPort
                privatePort
                ip
                isIpPublic
                type
              }}
            }}
          }}
        }}
        """

        # Wait until runtime.uptimeInSeconds becomes non-zero (or some threshold)
        while True:
            async with session.post(
                RUNPOD_ENDPOINT,
                headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
                json={"query": check_query}
            ) as resp:
                info = await resp.json()
                print("Poll raw:", json.dumps(info, indent=2))
                if "errors" in info:
                    raise RuntimeError(f"Poll error: {info['errors']}")
                pod_data = info.get("data", {}).get("pod")
                if pod_data:
                    rt = pod_data.get("runtime")
                    if rt:
                        uptime = rt.get("uptimeInSeconds")
                        if uptime is not None and uptime > 0:
                            print(f"Pod {pod_id} uptime: {uptime} s → assuming running")
                            break
            await asyncio.sleep(5)

        print(f"✅ Pod {pod_id} seems running (uptime > 0).")
        return pod_id

pod_id = await create_and_check_pod()


Create raw response: {
  "errors": [
    {
      "message": "String cannot represent a non string value: [\"11434/tcp\"]",
      "locations": [
        {
          "line": 10,
          "column": 20
        }
      ],
      "extensions": {
        "code": "GRAPHQL_VALIDATION_FAILED"
      }
    }
  ]
}


RuntimeError: Create error: [{'message': 'String cannot represent a non string value: ["11434/tcp"]', 'locations': [{'line': 10, 'column': 20}], 'extensions': {'code': 'GRAPHQL_VALIDATION_FAILED'}}]

In [14]:
import os, asyncio, aiohttp, json

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"
IMAGE_NAME = "runpod/ollama:latest"
GPU_TYPE = "A40"  # adjust based on gpuTypes query

async def create_and_check_pod():
    async with aiohttp.ClientSession() as session:
        query = f"""
        mutation {{
          podFindAndDeployOnDemand(input: {{
            name: "hello-ollama",
            imageName: "{IMAGE_NAME}",
            gpuTypeId: "{GPU_TYPE}",
            gpuCount: 1,
            containerDiskInGb: 20,
            volumeInGb: 0,
            ports: [{{ containerPort: 11434, protocol: TCP }}],
            env: [{{ key: "OLLAMA_HOST", value: "0.0.0.0" }}]
          }}) {{
            id
          }}
        }}
        """
        async with session.post(
            RUNPOD_ENDPOINT,
            headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            json={"query": query}
        ) as resp:
            data = await resp.json()
            print("Create raw response:", json.dumps(data, indent=2))
            if "errors" in data:
                raise RuntimeError(f"Create error: {data['errors']}")
            pod_info = data.get("data", {}).get("podFindAndDeployOnDemand")
            if not pod_info:
                raise RuntimeError(f"No pod info: {data}")
            pod_id = pod_info["id"]
            print(f"Pod created: {pod_id}")

        # Poll for readiness using uptimeInSeconds as a proxy for "running"
        check_query = f"""
        query {{
          pod(input: {{ podId: "{pod_id}" }}) {{
            id
            runtime {{
              uptimeInSeconds
              ports {{
                publicPort
                containerPort
                ip
              }}
            }}
          }}
        }}
        """
        while True:
            async with session.post(
                RUNPOD_ENDPOINT,
                headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
                json={"query": check_query}
            ) as resp:
                info = await resp.json()
                print("Poll raw:", json.dumps(info, indent=2))
                if "errors" in info:
                    raise RuntimeError(f"Poll error: {info['errors']}")
                runtime = info.get("data", {}).get("pod", {}).get("runtime")
                if runtime and runtime.get("uptimeInSeconds", 0) > 0:
                    print(f"✅ Pod {pod_id} is running.")
                    break
            await asyncio.sleep(5)

        return pod_id

pod_id = await create_and_check_pod()


Exception ignored in: <coroutine object main at 0x00000258B76C3880>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object main at 0x00000258B76C3880>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'
Exception ignored in: <coroutine object main at 0x00000258B7D843A0>
Traceback (most recent call last):
  File "<string>", line 1, in <lambda>
KeyError: '__import__'


Create raw response: {
  "errors": [
    {
      "message": "String cannot represent a non string value: [{containerPort: 11434, protocol: TCP}]",
      "locations": [
        {
          "line": 10,
          "column": 20
        }
      ],
      "extensions": {
        "code": "GRAPHQL_VALIDATION_FAILED"
      }
    }
  ]
}


RuntimeError: Create error: [{'message': 'String cannot represent a non string value: [{containerPort: 11434, protocol: TCP}]', 'locations': [{'line': 10, 'column': 20}], 'extensions': {'code': 'GRAPHQL_VALIDATION_FAILED'}}]

In [17]:
import os, asyncio, aiohttp, json

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"
IMAGE_NAME = "runpod/ollama:latest"
GPU_TYPE = "A40"  # Make sure this matches what `gpuTypes` returns

async def create_and_check_pod():
    async with aiohttp.ClientSession() as session:
        mutation = f"""
        mutation {{
          podFindAndDeployOnDemand(input: {{
            name: "hello-ollama",
            imageName: "{IMAGE_NAME}",
            gpuTypeId: "{GPU_TYPE}",
            gpuCount: 1,
            containerDiskInGb: 20,
            volumeInGb: 0,
            ports: "11434/tcp",   # correct format
            env: [{{ key: "OLLAMA_HOST", value: "0.0.0.0" }}]
          }}) {{
            id
          }}
        }}
        """
        async with session.post(
            RUNPOD_ENDPOINT,
            headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
            json={"query": mutation}
        ) as resp:
            create_resp = await resp.json()
            print("Create response:", json.dumps(create_resp, indent=2))
            if "errors" in create_resp:
                raise RuntimeError(f"Create failed: {create_resp['errors']}")
            pod_info = create_resp.get("data", {}).get("podFindAndDeployOnDemand")
            if not pod_info or "id" not in pod_info:
                raise RuntimeError(f"No pod id returned: {create_resp}")
            pod_id = pod_info["id"]
            print(f"Pod created with id: {pod_id}")

        # Poll for runtime uptime etc. as readiness check
        check_query = f"""
        query {{
          pod(input: {{ podId: "{pod_id}" }}) {{
            id
            runtime {{
              uptimeInSeconds
              ports {{
                publicPort
                privatePort
                isIpPublic
                type
                ip
              }}
            }}
          }}
        }}
        """
        while True:
            await asyncio.sleep(5)
            async with session.post(
                RUNPOD_ENDPOINT,
                headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
                json={"query": check_query}
            ) as resp:
                poll_resp = await resp.json()
                print("Poll response:", json.dumps(poll_resp, indent=2))
                if "errors" in poll_resp:
                    raise RuntimeError(f"Poll error: {poll_resp['errors']}")
                pod_data = poll_resp.get("data", {}).get("pod", {})
                runtime = pod_data.get("runtime")
                if runtime and runtime.get("uptimeInSeconds", 0) > 0:
                    print(f"✅ Pod {pod_id} is up (uptimeInSeconds: {runtime['uptimeInSeconds']})")
                    break

        return pod_id

# In Jupyter notebook, use:
import nest_asyncio; nest_asyncio.apply()
pod_id = await create_and_check_pod()


Create response: {
  "errors": [
    {
      "message": "There are no longer any instances available with the requested specifications. Please refresh and try again.",
      "path": [
        "podFindAndDeployOnDemand"
      ],
      "extensions": {
        "code": "RUNPOD"
      }
    }
  ],
  "data": {
    "podFindAndDeployOnDemand": null
  }
}


RuntimeError: Create failed: [{'message': 'There are no longer any instances available with the requested specifications. Please refresh and try again.', 'path': ['podFindAndDeployOnDemand'], 'extensions': {'code': 'RUNPOD'}}]

In [18]:
import aiohttp, asyncio, os, json

async def gpu_types():
    async with aiohttp.ClientSession() as s:
        q = "query { gpuTypes { id displayName memoryInGb isAvailable } }"
        async with s.post(
            "https://api.runpod.io/graphql",
            headers={"Authorization": f"Bearer {os.getenv('RUNPOD_API_KEY')}"},
            json={"query": q}
        ) as r:
            print(json.dumps(await r.json(), indent=2))

await gpu_types()


{
  "errors": [
    {
      "message": "Cannot query field \"isAvailable\" on type \"GpuType\".",
      "locations": [
        {
          "line": 1,
          "column": 46
        }
      ],
      "extensions": {
        "code": "GRAPHQL_VALIDATION_FAILED"
      }
    }
  ]
}


In [ ]:
# Cell 1: Imports & Config
import os, json, asyncio, aiohttp
from pathlib import Path
from datetime import datetime
import pandas as pd

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"

MODELS = ["llama3", "mistral"]
BENCHMARKS = ["bench1", "bench2"]
GPU_TYPE = "NVIDIA RTX A6000"
IMAGE_NAME = "runpod/ollama:latest"


# Single Model Per Pod

In [ ]:
# Cell 2: Runpod helpers
async def gql(session, query):
    async with session.post(
        RUNPOD_ENDPOINT,
        headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
        json={"query": query},
    ) as r:
        resp = await r.json()
        if "errors" in resp:
            raise RuntimeError(resp["errors"])
        return resp["data"]

async def create_pod(session, name):
    query = f"""
    mutation {{
      podCreate(input: {{
        name: "{name}",
        imageName: "{IMAGE_NAME}",
        gpuTypeId: "{GPU_TYPE}",
        containerDiskInGb: 20,
        volumeInGb: 10,
        ports: [{{containerPort:11434, hostPort:11434, protocol:TCP}}],
        env: [{{key:"OLLAMA_HOST", value:"0.0.0.0"}}]
      }}) {{ id status }}
    }}
    """
    return await gql(session, query)

async def get_pod_url(session, pod_id):
    q = f"""
    query {{
      pod(id:"{pod_id}") {{
        runtime {{ host ports {{ hostPort isPublic }} }}
        status
      }}
    }}
    """
    return await gql(session, q)

async def stop_pod(session, pod_id):
    q = f'mutation {{ podTerminate(id:"{pod_id}") }}'
    return await gql(session, q)


In [ ]:
# Cell 3: Ollama helpers
async def ollama_call(session, base_url, endpoint, payload):
    async with session.post(f"{base_url}{endpoint}", json=payload) as r:
        return await r.json()

async def pull_model(session, base_url, model):
    return await ollama_call(session, base_url, "/api/pull", {"model": model})

async def run_benchmark(session, base_url, model, benchmark):
    # Replace with your real benchmark call
    # Example: pass prompts to /api/generate
    prompt = f"Benchmark {benchmark}: test input"
    return await ollama_call(session, base_url, "/api/generate",
                             {"model": model, "prompt": prompt})


In [ ]:
# Cell 4: Orchestrate one model on one pod
async def run_model_benchmarks(model):
    async with aiohttp.ClientSession() as session:
        # 1. Create pod
        pod_name = f"ollama-{model}-{int(datetime.now().timestamp())}"
        pod = await create_pod(session, pod_name)
        pod_id = pod["podCreate"]["id"]

        # 2. Wait for running status and get URL
        print(f"[{model}] waiting for pod to be ready…")
        url = None
        while True:
            status = await get_pod_url(session, pod_id)
            s = status["pod"]["status"]
            if s == "RUNNING":
                port = status["pod"]["runtime"]["ports"][0]["hostPort"]
                host = status["pod"]["runtime"]["host"]
                url = f"https://{host}-{port}.proxy.runpod.net"
                break
            await asyncio.sleep(10)

        # 3. Pull model
        print(f"[{model}] pulling model…")
        await pull_model(session, url, model)

        # 4. Run benchmarks
        results = []
        for bench in BENCHMARKS:
            print(f"[{model}] running {bench}")
            r = await run_benchmark(session, url, model, bench)
            results.append({"model": model, "benchmark": bench, "result": r})

        # 5. Terminate pod
        await stop_pod(session, pod_id)
        return results


In [ ]:
# Cell 5: Launch all models in parallel
async def main():
    tasks = [run_model_benchmarks(m) for m in MODELS]
    nested_results = await asyncio.gather(*tasks)
    # Flatten and store
    flat = [item for sub in nested_results for item in sub]
    df = pd.DataFrame(flat)
    df.to_csv("ollama_benchmark_results.csv", index=False)
    return df

df_results = asyncio.run(main())
df_results.head()


# Method 2: Multiple Models on a Pod

In [ ]:
import os, asyncio, aiohttp, json, pandas as pd
from datetime import datetime

RUNPOD_API_KEY = os.getenv("RUNPOD_API_KEY")
RUNPOD_ENDPOINT = "https://api.runpod.io/graphql"

# Model groups: several models per pod
MODEL_GROUPS = [
    ["llama3-8b", "mistral-7b", "gemma-2b"],
    ["llama3-70b", "mixtral-8x7b"],      # might need 2–3 GPUs
]

# SysEngBench tasks to run with lm-eval
# BENCHMARKS = ["sysengbench_mcq", "sysengbench_osq"]
BENCHMARKS = ["sysengbench", "sysengbench"]

GPU_TYPE = "NVIDIA A40"
IMAGE_NAME = "runpod/ollama:latest"


In [2]:
# GraphQL helpers
async def gql(session, query):
    async with session.post(
        RUNPOD_ENDPOINT,
        headers={"Authorization": f"Bearer {RUNPOD_API_KEY}"},
        json={"query": query}
    ) as r:
        data = await r.json()
        if "errors" in data: raise RuntimeError(data["errors"])
        return data["data"]

async def create_pod(session, name, gpu_count):
    # choose correct GPUType (single type but set count)
    q = f"""
    mutation {{
      podCreate(input: {{
        name: "{name}",
        imageName: "{IMAGE_NAME}",
        gpuTypeId: "{GPU_TYPE}",
        gpuCount: {gpu_count},
        containerDiskInGb: 40,
        volumeInGb: 20,
        ports: [{{containerPort:11434, hostPort:11434, protocol:TCP}}],
        env: [{{key:"OLLAMA_HOST", value:"0.0.0.0"}}],
        containerStartCommand: "pip install lm-eval && ollama serve"
      }}) {{
        id status
      }}
    }}
    """
    return await gql(session, q)

async def get_pod_url(session, pod_id):
    q = f"""
    query {{
      pod(id:"{pod_id}") {{
        status
        runtime {{ host ports {{ hostPort isPublic }} }}
      }}
    }}
    """
    return await gql(session, q)

async def stop_pod(session, pod_id):
    q = f'mutation {{ podTerminate(id:"{pod_id}") }}'
    return await gql(session, q)


In [3]:
# Ollama & lm-eval utilities
async def ollama_call(session, base_url, endpoint, payload):
    async with session.post(f"{base_url}{endpoint}", json=payload) as r:
        return await r.json()

async def pull_model(session, base_url, model):
    return await ollama_call(session, base_url, "/api/pull", {"model": model})

async def run_sysengbench(session, base_url, model, benchmark):
    # Run lm-eval inside the pod using Ollama backend
    cmd = (
        f"lm_eval --model ollama "
        f"--model_args model={model},base_url={base_url} "
        f"--tasks {benchmark} --output_path results/{model}_{benchmark}.json"
    )
    return await ollama_call(session, base_url, "/api/generate",
                             {"model": model, "prompt": f"run shell: {cmd}"})


In [5]:
# Run all models for one pod
async def run_group(group, gpu_count):
    async with aiohttp.ClientSession() as session:
        pod_name = f"ollama-syseng-{group[0]}-{int(datetime.now().timestamp())}"
        pod = await create_pod(session, pod_name, gpu_count)
        pod_id = pod["podCreate"]["id"]

        # Wait for pod ready
        print(f"[{group}] waiting for pod...")
        while True:
            info = await get_pod_url(session, pod_id)
            if info["pod"]["status"] == "RUNNING":
                port = info["pod"]["runtime"]["ports"][0]["hostPort"]
                host = info["pod"]["runtime"]["host"]
                url = f"https://{host}-{port}.proxy.runpod.net"
                break
            await asyncio.sleep(10)

        # Pull all models for this group
        for m in group:
            print(f"[{group}] pulling {m}")
            await pull_model(session, url, m)

        # Run benchmarks
        results = []
        for m in group:
            for bench in BENCHMARKS:
                print(f"[{m}] running {bench}")
                r = await run_sysengbench(session, url, m, bench)
                results.append({"pod": pod_name, "model": m,
                                "benchmark": bench, "result": r})

        # Terminate pod
        await stop_pod(session, pod_id)
        return results


In [ ]:
# Parallel execution of all groups
async def main():
    tasks = []
    for group in MODEL_GROUPS:
        # Determine GPU count: if group has heavy models, request 2 or 3 GPUs
        gpu_count = 3 if any("70b" in m for m in group) else 1
        tasks.append(run_group(group, gpu_count))
    nested = await asyncio.gather(*tasks)
    flat = [item for sub in nested for item in sub]
    df = pd.DataFrame(flat)
    df.to_csv("sysengbench_results.csv", index=False)
    return df

df_results = asyncio.run(main())

# df_results = await main()
df_results.head()


RuntimeError: [{'message': 'Cannot query field "podCreate" on type "Mutation". Did you mean "podReset", "podResume", or "teamCreate"?', 'locations': [{'line': 3, 'column': 7}], 'extensions': {'code': 'GRAPHQL_VALIDATION_FAILED'}}]

# Method 3: Systematically pull back the file via HTTP server

In [ ]:
async def create_pod(session, name, gpu_count):
    q = f"""
    mutation {{
      podCreate(input: {{
        name: "{name}",
        imageName: "{IMAGE_NAME}",
        gpuTypeId: "{GPU_TYPE}",
        gpuCount: {gpu_count},
        containerDiskInGb: 40,
        volumeInGb: 20,
        ports: [
          {{containerPort:11434, hostPort:11434, protocol:TCP}},
          {{containerPort:8080, hostPort:8080, protocol:TCP}}
        ],
        env: [{{key:"OLLAMA_HOST", value:"0.0.0.0"}}],
        containerStartCommand: "pip install lm-eval && mkdir -p results && python3 -m http.server 8080 --directory results & ollama serve"
      }}) {{
        id status
      }}
    }}
    """
    return await gql(session, q)


In [ ]:
cmd = (
    f"lm_eval --model ollama "
    f"--model_args model={model},base_url={base_url} "
    f"--tasks {benchmark} --output_path results/{model}_{benchmark}.json"
)


In [ ]:
from pathlib import Path

async def fetch_results_from_pod(session, base_url, local_dir="downloaded_results"):
    Path(local_dir).mkdir(exist_ok=True)
    # simple listing: since http.server has no directory API,
    # we rely on known filenames or have the notebook track them
    # (we know the models and benchmarks)
    for m in group:
        for bench in BENCHMARKS:
            fname = f"{m}_{bench}.json"
            url = f"{base_url}/{fname}"
            async with session.get(url) as r:
                if r.status == 200:
                    with open(Path(local_dir) / fname, "wb") as f:
                        f.write(await r.read())
                    print(f"Downloaded {fname}")
                else:
                    print(f"Warning: {fname} not found (status {r.status})")


In [ ]:
async def run_group(group, gpu_count):
    async with aiohttp.ClientSession() as session:
        # … existing pod creation / model pulls / benchmarks …

        # Fetch results back to laptop
        await fetch_results_from_pod(session, f"https://{host}-8080.proxy.runpod.net")

        # Now safe to stop pod
        await stop_pod(session, pod_id)
